# Reconstruct a synthetic air shower

This notebook is a truth-controlled test of the plane-front direction mathematics. We choose a known sky-arrival direction, generate the station times implied by that direction, reconstruct the direction using only station positions, times, and timing uncertainties, and compare the recovered result with the hidden truth.

## 1. Imports, units, and reproducibility

In [12]:
import numpy as np
import pandas as pd
from scipy.optimize import least_squares, curve_fit

C_M_PER_NS = 0.299792458
RNG_SEED = 20260829
rng = np.random.default_rng(RNG_SEED)

## 2. Define a synthetic detector array

In [13]:
station_coordinates_m = np.array([
    
        [-1500.0, -1500.0, 35.0],
        [0.0, -1500.0, 22.0],
        [1500.0, -1500.0, 45.0],
        [-1500.0, 0.0, 18.0],
        [0.0, 0.0, 30.0],
        [1500.0, 0.0, 12.0],
        [-1500.0, 1500.0, 55.0],
        [0.0, 1500.0, 42.0],
        [1500.0, 1500.0, 60.0],
        [-750.0, -750.0, 28.0],
        [750.0, -750.0, 20.0],
        [-750.0, 750.0, 46.0],
        [750.0, 750.0, 38.0]
        ],
        dtype = float,
)

timing_uncertainties_ns = np.array(
    [
        8.0,
        12.0,
        10.0,
        15.0,
        7.0,
        18.0,
        11.0,
        9.0,
        14.0,
        8.0,
        16.0,
        10.0,
        13.0,
    ],
    dtype=float,
)

if station_coordinates_m.ndim != 2 or station_coordinates_m.shape[1] != 3:
    raise ValueError("Station coordinates must have shape (N, 3) where N is the number of stations.")

if len(timing_uncertainties_ns) != len(station_coordinates_m):
    raise ValueError("Every station must have exactly one timing uncertainty.")

if not np.isfinite(station_coordinates_m).all():
    raise ValueError("Station coordinates must must be finite.")

if not np.isfinite(timing_uncertainties_ns).all():
    raise ValueError("Timing uncertainties must be finite.")

if (timing_uncertainties_ns <= 0.0).any():
    raise ValueError("Timing uncertainties must be positive.")

station_names = [f"S{number:02d}" for number in range(1, len(station_coordinates_m) + 1)]

synthetic_stations = pd.DataFrame(
    station_coordinates_m,
    columns = ["x_m", "y_m", "z_m"]
)

synthetic_stations.insert(0, "station_name", station_names)
synthetic_stations["dt_ns"] = timing_uncertainties_ns

synthetic_stations

,station_name,x_m,y_m,z_m,dt_ns
0,S01,-1500.0,-1500.0,35.0,8.0
1,S02,0.0,-1500.0,22.0,12.0
2,S03,1500.0,-1500.0,45.0,10.0
3,S04,-1500.0,0.0,18.0,15.0
4,S05,0.0,0.0,30.0,7.0
5,S06,1500.0,0.0,12.0,18.0
6,S07,-1500.0,1500.0,55.0,11.0
7,S08,0.0,1500.0,42.0,9.0
8,S09,1500.0,1500.0,60.0,14.0
9,S10,-750.0,-750.0,28.0,8.0


## 3. Choose the hidden true shower direction

In [14]:
TRUE_ZENITH_DEG = 42.0
TRUE_AZIMUTH_DEG = 55.0
TRUE_ORIGIN_TIME_NS = 100000.0

theta_true_rad = np.deg2rad(TRUE_ZENITH_DEG)
phi_true_rad = np.deg2rad(TRUE_AZIMUTH_DEG)

sky_direction_true = np.array(
    [
        np.sin(theta_true_rad) * np.cos(phi_true_rad),
        np.sin(theta_true_rad) * np.sin(phi_true_rad),
        np.cos(theta_true_rad)
    ],
    dtype = float
)

propagation_direction_true = -sky_direction_true

direction_norm = np.linalg.norm(sky_direction_true)

if not np.isclose(
    direction_norm,
    1.0,
    rtol = 0.0,
    atol = 1e-12
):
    raise ValueError(f"The true sky direction is not a unit vector. Norm = {direction_norm}")

if propagation_direction_true[2] >= 0.0:
    raise ValueError("The synthetic shower must propagate downwards.")

print(f"True sky vector:         {sky_direction_true}")
print(f"True propagation vector: {propagation_direction_true}")
print(f"Sky-vector norm:         {direction_norm:.15f}")

True sky vector:         [0.38379755 0.5481197  0.74314483]
True propagation vector: [-0.38379755 -0.5481197  -0.74314483]
Sky-vector norm:         1.000000000000000


## 4. Generate ideal station arrival times

In [15]:
POSITION_COLUMNS = ["x_m", "y_m", "z_m"]

positions_m = synthetic_stations.loc[:, POSITION_COLUMNS].to_numpy(dtype = float, copy = True)

geometric_delay_ns = (positions_m @ propagation_direction_true / C_M_PER_NS)

synthetic_stations["t_exact_ns"] = (
    TRUE_ORIGIN_TIME_NS + geometric_delay_ns
)

synthetic_stations["t_observed_ns"] = (
    synthetic_stations["t_exact_ns"]
)

synthetic_stations["relative_time_ns"] = (
    synthetic_stations["t_observed_ns"]
    - synthetic_stations["t_observed_ns"].min()
)

synthetic_stations.loc[
    :,
    [
        "station_name",
        "x_m",
        "y_m",
        "z_m",
        "dt_ns",
        "t_exact_ns",
        "relative_time_ns",
    ],
]

,station_name,x_m,y_m,z_m,dt_ns,t_exact_ns,relative_time_ns
0,S01,-1500.0,-1500.0,35.0,8.0,104576.051777,9387.595664
1,S02,0.0,-1500.0,22.0,12.0,102687.960782,7499.504669
2,S03,1500.0,-1500.0,45.0,10.0,100710.630672,5522.174559
3,S04,-1500.0,0.0,18.0,15.0,101875.696673,6687.240560
4,S05,0.0,0.0,30.0,7.0,99925.634071,4737.177958
5,S06,1500.0,0.0,12.0,18.0,98049.937397,2861.481284
6,S07,-1500.0,1500.0,55.0,11.0,99041.482896,3853.026783
7,S08,0.0,1500.0,42.0,9.0,97153.391901,1964.935788
8,S09,1500.0,1500.0,60.0,14.0,95188.456113,0.000000
9,S10,-750.0,-750.0,28.0,8.0,102261.997813,7073.541700


## 5. Reconstruct the propagation vector

In [16]:
times_ns = synthetic_stations["t_observed_ns"].to_numpy(
    dtype = float,
    copy = True
)

dt_ns = synthetic_stations["dt_ns"].to_numpy(
    dtype = float,
    copy = True
)

weights = 1.0 /dt_ns**2

position_reference_m = np.average(
    positions_m,
    axis = 0,
    weights =  weights
)

time_reference_ns = np.average(
    times_ns,
    weights = weights
)

delta_positions_m = positions_m - position_reference_m
delta_time_ns = times_ns - time_reference_ns

design_matrix_ns = delta_positions_m / C_M_PER_NS

sqrt_weights = np.sqrt(weights)

weighted_design_matrix = (
    design_matrix_ns * sqrt_weights[:, np.newaxis]
)

weighted_time_vector = delta_time_ns * sqrt_weights

(
    propagation_linear,
    residual_sum_squares,
    matrix_rank,
    singular_values
) = np.linalg.lstsq(
    weighted_design_matrix,
    weighted_time_vector,
    rcond = None
)

if matrix_rank < 3:
    raise np.linalg.LinAlgError(" The station geometry can't determine all three direction components.")

linear_norm = float(np.linalg.norm(propagation_linear))

if not np.isfinite(linear_norm) or linear_norm <= 0.0:
    raise ValueError("The linear direction estimate has an invalid norm.")

propagation_initial = propagation_linear / linear_norm

if propagation_initial[2] >= 0.0:
    raise ValueError("The initial result propagates upward; check the sign convention.")

print(f"Matrix rank:                  {matrix_rank}")
print(f"Linear propagation estimate: {propagation_linear}")
print(f"Linear-vector norm:           {linear_norm:.15f}")
print(f"Linear residual output:       {residual_sum_squares}")
print(f"Singular values:              {singular_values}")

Matrix rank:                  3
Linear propagation estimate: [-0.38379755 -0.5481197  -0.74314483]
Linear-vector norm:           1.000000000000004
Linear residual output:       [2.97443107e-24]
Singular values:              [1296.91589322 1141.51106063   11.24475085]


## 6. Convert the fitted vector into sky angles

In [17]:
def propagation_from_sky_angles(theta_rad: float, phi_rad: float) -> np.ndarray:
    sky_direction = np.array(
        [
            np.sin(theta_rad) * np.cos(phi_rad),
            np.sin(theta_rad) * np.sin(phi_rad),
            np.cos(theta_rad)
        ],
        dtype = float
    ) 

    return -sky_direction

def standardized_timing_residuals(parameters: np.ndarray, station_positions_m: np.ndarray, observed_times_ns: np.ndarray, 
                                  timing_uncertainties_ns: np.ndarray, reference_position_m: np.ndarray) -> np.ndarray:
    reference_time_ns, theta_rad, phi_rad =  parameters
    propagation_direction = propagation_from_sky_angles(theta_rad, phi_rad)
    predicted_times_ns = (reference_time_ns + (station_positions_m - reference_position_m) @ propagation_direction / C_M_PER_NS)

    return (observed_times_ns - predicted_times_ns) / timing_uncertainties_ns

sky_initial = -propagation_initial

theta_initial_rad = np.arccos(np.clip(sky_initial[2], -1.0, 1.0))

phi_initial_rad = np.arctan2(sky_initial[1], sky_initial[0])

initial_parameters = np.array(
    [
        time_reference_ns,
        theta_initial_rad,
        phi_initial_rad
    ],
    dtype = float
)

fit = least_squares(
    standardized_timing_residuals,
    x0 = initial_parameters,
    args = (
        positions_m,
        times_ns,
        dt_ns,
        position_reference_m
    ),
    bounds = (
        np.array([-np.inf, 0.0, -np.inf]),
        np.array([np.inf, np.pi / 2.0, np.inf])
    ),
    method = "trf",
    x_scale = "jac"
)

if not fit.success: raise RuntimeError(f"The constrained fit failed: {fit.message}")

(
    fitted_reference_time_ns,
    theta_fit_rad,
    phi_fit_unwrapped_rad
) = fit.x

phi_fit_rad = phi_fit_unwrapped_rad % (2.0 * np.pi)

propagation_direction_fit = propagation_from_sky_angles(theta_fit_rad, phi_fit_rad)

sky_direction_fit = -propagation_direction_fit


## 7. Evaluate timing and angular errors

In [20]:
def angular_separation_deg(first_vector: np.ndarray, second_vector: np.ndarray) -> float:
    first_norm = float(np.linalg.norm(first_vector))
    second_norm =  float(np.linalg.norm(second_vector))

    if first_norm <= 0.0 or second_norm <= 0.0:
        raise ValueError("Direction vectors must have nonzero length.")

    first_unit = first_vector / first_norm
    second_unit = second_vector / second_norm

    cosine_separation = np.clip(np.dot(first_unit, second_unit), -1.0, 1.0)

    return float(np.rad2deg(np.arccos(cosine_separation)))


predicted_times_ns = (fitted_reference_time_ns + (positions_m - position_reference_m) @ propagation_direction_fit / C_M_PER_NS)

timing_residuals_ns = times_ns - predicted_times_ns
standardized_residuals = timing_residuals_ns / dt_ns

theta_fit_deg = float(np.rad2deg(theta_fit_rad))
phi_fit_deg = float(np.rad2deg(phi_fit_rad))

angular_error_deg = angular_separation_deg(sky_direction_fit, sky_direction_true)

chi_square = float(np.sum(standardized_residuals**2))

degrees_of_freedom = len(times_ns) - 3
reduced_chi_square = chi_square / degrees_of_freedom

fitted_origin_time_ns = (fitted_reference_time_ns - position_reference_m @ propagation_direction_fit / C_M_PER_NS)

diagnostics = synthetic_stations.loc[:, ["station_name",
                                         "dt_ns",
                                         "t_observed_ns"]].copy()

diagnostics["t_predicted_ns"] = predicted_times_ns
diagnostics["residual_ns"] = timing_residuals_ns
diagnostics["standardized_residual"] = standardized_residuals

print(f"True zenith:              {TRUE_ZENITH_DEG:.8f} deg")
print(f"Fitted zenith:            {theta_fit_deg:.8f} deg")
print(f"True azimuth:             {TRUE_AZIMUTH_DEG:.8f} deg")
print(f"Fitted azimuth:           {phi_fit_deg:.8f} deg")
print(f"Angular error:            {angular_error_deg:.12f} deg")
print(f"True origin time:         {TRUE_ORIGIN_TIME_NS:.8f} ns")
print(f"Fitted origin time:       {fitted_origin_time_ns:.8f} ns")
print(
    "Maximum timing residual: "
    f"{np.max(np.abs(timing_residuals_ns)):.12e} ns"
)
print(f"Chi-square:               {chi_square:.12e}")
print(f"Degrees of freedom:       {degrees_of_freedom}")
print(f"Reduced chi-square:       {reduced_chi_square:.12e}")

diagnostics


True zenith:              42.00000000 deg
Fitted zenith:            42.00000000 deg
True azimuth:             55.00000000 deg
Fitted azimuth:           55.00000000 deg
Angular error:            0.000000000000 deg
True origin time:         100000.00000000 ns
Fitted origin time:       100000.00000000 ns
Maximum timing residual: 2.910383045673e-11 ns
Chi-square:               1.601885090418e-23
Degrees of freedom:       10
Reduced chi-square:       1.601885090418e-24


,station_name,dt_ns,t_observed_ns,t_predicted_ns,residual_ns,standardized_residual
0,S01,8.0,104576.051777,104576.051777,1.455192e-11,1.818989e-12
1,S02,12.0,102687.960782,102687.960782,0.000000e+00,0.000000e+00
2,S03,10.0,100710.630672,100710.630672,0.000000e+00,0.000000e+00
3,S04,15.0,101875.696673,101875.696673,0.000000e+00,0.000000e+00
4,S05,7.0,99925.634071,99925.634071,0.000000e+00,0.000000e+00
5,S06,18.0,98049.937397,98049.937397,-1.455192e-11,-8.084397e-13
6,S07,11.0,99041.482896,99041.482896,-1.455192e-11,-1.322901e-12
7,S08,9.0,97153.391901,97153.391901,-1.455192e-11,-1.616879e-12
8,S09,14.0,95188.456113,95188.456113,-2.910383e-11,-2.078845e-12
9,S10,8.0,102261.997813,102261.997813,0.000000e+00,0.000000e+00


## 8. Add controlled timing noise and repeat

## 9. Record conclusions and implementation requirements